<a href="https://colab.research.google.com/github/itzz-soorya/Agent_with_langchain/blob/main/agent_with_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community pypdf langchain-huggingface sentence-transformers chromadb langchain-core langchain-google-genai langchain==0.1.16 langchain-community==0.0.34

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("technical_terms.pdf")
documents=loader.load()

In [ ]:
print(f"loaded {len(documents)} pages from the pdf")

loaded 6 pages from the pdf


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

chunks=text_splitter.split_documents(documents)

In [ ]:
print(len(chunks))

40


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
print("vector store created succesfully")

vector store created succesfully


In [ ]:
from langchain_core.tools import Tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def retrieve_technical_term(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    return "\n".join([doc.page_content for doc in docs])

retrieval_tool = Tool(
    name="TechnicalTermRetriever",
    func=retrieve_technical_term,
    description="Use this tool to retrieve definitions of technical terms from the document"
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
os.environ["GOOGLE_API_KEY"]="replays ur api key"


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical dictionary assistant. Use tools when needed."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])



In [ ]:
from langchain.agents import initialize_agent, AgentType


agent = initialize_agent(
    tools=[retrieval_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False
)


In [ ]:
agent.run("define api in bullet points")


'* API stands for Application Programming Interface.\n* It allows different software applications to communicate with each other and share data or services.'